In [ ]:
import glob, os, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# --- locate data ---
def find(name):
    return glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]

tr = pd.read_csv(find("student_dataset.csv"))
te = pd.read_csv(find("test.csv"))
ss = pd.read_csv(find("sample_submission.csv"))

NUM = ["ambient_temp", "process_temp", "coolant_flow", "shaft_speed", "drive_torque",
       "power_load", "component_wear", "thermal_stress", "mechanical_load", "vibration_index"]
ENG = ["dT", "dT_cool", "pw", "wear_tq", "wear_vib", "ts_cool", "ml_wear", "tq_speed_ratio"]
LOGC = ["component_wear", "power_load", "mechanical_load", "vibration_index", "thermal_stress",
        "dT_cool", "ts_cool", "wear_tq", "wear_vib", "ml_wear"]

def fe(d):
    d = d.copy()
    e = 1e-6
    for c in NUM:
        d[c + "_na"] = d[c].isna().astype(int)
    d["dT"] = d.process_temp - d.ambient_temp
    d["dT_cool"] = d.dT / (d.coolant_flow + e)
    d["pw"] = d.shaft_speed * d.drive_torque / 9550
    d["wear_tq"] = d.component_wear * d.drive_torque
    d["wear_vib"] = d.component_wear * d.vibration_index
    d["ts_cool"] = d.thermal_stress / (d.coolant_flow + e)
    d["ml_wear"] = d.mechanical_load * d.component_wear
    d["tq_speed_ratio"] = d.drive_torque / (d.shaft_speed + e)
    for c in LOGC:
        d["log_" + c] = np.log1p(d[c].clip(lower=0))
    return d

def build(C=0.3):
    cols = NUM + ENG + ["log_" + c for c in LOGC]
    pre = ColumnTransformer([
        ("n", Pipeline([("imp", SimpleImputer(strategy="median")),
                        ("sc", StandardScaler()),
                        ("poly", PolynomialFeatures(2, include_bias=False)),
                        ("sc2", StandardScaler())]), cols),
        ("c", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                        ("o", OneHotEncoder(handle_unknown="ignore"))]), ["unit_type"]),
        ("na", "passthrough", [c + "_na" for c in NUM]),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(C=C, max_iter=5000))])

X, y = fe(tr), tr["condition"]

# --- cross-validation ---
cv = StratifiedKFold(5, shuffle=True, random_state=42)
scores = cross_val_score(build(), X, y, cv=cv, scoring="accuracy", n_jobs=-1)
print("CV accuracy:", scores.mean().round(4), "+/-", scores.std().round(4))

# --- fit on all data and predict ---
model = build().fit(X, y)
pred = model.predict(fe(te))

sub = ss[["unit_id"]].merge(pd.DataFrame({"unit_id": te.unit_id, "condition": pred}),
                            on="unit_id", how="left")
assert sub.condition.notna().all() and len(sub) == len(ss)
sub["condition"] = sub.condition.astype(int)
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.condition.value_counts(normalize=True))
print("Submission file successfully generated at '/kaggle/working/submission.csv'!")